# Football Forecasting & Simulation Lab

This notebook demonstrates the complete modeling workflow: synthetic data validation, walk-forward forecasting, probability diagnostics, exact-score predictions, and Monte Carlo league simulation.

> The default results are explicitly **synthetic**. The final section can download real Premier League and La Liga data when `RUN_REAL_DATA` is changed to `True`. Run Jupyter from the repository root so all paths resolve correctly.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

from football_lab.data.synthetic import (
    create_simulation_snapshot,
    generate_synthetic_league,
)
from football_lab.evaluation.backtest import benchmark_models
from football_lab.evaluation.metrics import calibration_table
from football_lab.models import (
    DixonColesModel,
    IndependentPoissonModel,
    LeagueRateBaseline,
)
from football_lab.simulation.league import current_table, simulate_league

pd.set_option('display.max_columns', 30)
REPORT_DIR = Path('reports/notebook')
REPORT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Generate the zero-setup dataset

The generator creates three deterministic double round-robin seasons from hidden team strengths. It exists to verify the entire pipeline without downloading or redistributing third-party data.

In [ ]:
matches = generate_synthetic_league(n_teams=10, n_seasons=3, seed=42)
print(f'{len(matches):,} matches | {matches.season.nunique()} seasons | {matches.home_team.nunique()} teams')
display(matches.head())
display(
    matches.groupby('season').agg(
        matches=('date', 'size'),
        home_goals=('home_goals', 'mean'),
        away_goals=('away_goals', 'mean'),
    ).round(3)
)

## 2. Walk-forward model benchmark

Each model is refitted using only earlier matches. The backtester mechanically checks that every training cutoff is strictly before the predicted match date. Lower log loss, Brier score, and RPS are better.

In [ ]:
model_factories = {
    'league_rate_baseline': lambda: LeagueRateBaseline(),
    'independent_poisson': lambda: IndependentPoissonModel(),
    'dixon_coles': lambda: DixonColesModel(),
}

metrics, predictions = benchmark_models(
    matches,
    model_factories,
    initial_train_matches=100,
    refit_every=50,
)
display(metrics.round(4))
metrics.to_csv(REPORT_DIR / 'synthetic_benchmark_metrics.csv', index=False)

In [ ]:
plot_data = metrics.set_index('model')[
    ['log_loss', 'brier_score', 'ranked_probability_score']
]
axes = plot_data.plot.bar(subplots=True, figsize=(11, 8), legend=False, rot=20)
for axis in axes:
    axis.grid(axis='y', alpha=0.25)
plt.suptitle('Synthetic walk-forward benchmark — lower is better', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
reliability = calibration_table(predictions['dixon_coles'])
display(reliability.round(4))

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], '--', color='grey', label='Perfect calibration')
ax.plot(
    reliability['mean_confidence'],
    reliability['observed_accuracy'],
    marker='o',
    label='Dixon–Coles',
)
ax.set(xlabel='Mean predicted confidence', ylabel='Observed accuracy', xlim=(0, 1), ylim=(0, 1))
ax.set_title('Reliability of the most likely 1X2 outcome')
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 3. Inspect one match forecast

The model produces one normalized score matrix. Expected goals, exact scores, and 1X2 probabilities are all derived from that same object.

In [ ]:
completed, remaining_fixtures = create_simulation_snapshot(matches)
model = DixonColesModel().fit(completed)

home_team = 'Northbridge FC'
away_team = 'Riverside Athletic'
forecast = model.predict_match(home_team, away_team)
display(pd.Series(forecast, name=f'{home_team} vs {away_team}').to_frame())

score_matrix = model.score_matrix(home_team, away_team)
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(score_matrix, cmap='Blues', origin='lower')
ax.set(xlabel=f'{away_team} goals', ylabel=f'{home_team} goals')
ax.set_title('Exact-score probability matrix')
fig.colorbar(image, ax=ax, label='Probability')
plt.show()

top_scorelines = (
    pd.DataFrame(
        [
            {'score': f'{home}-{away}', 'probability': score_matrix[home, away]}
            for home in range(score_matrix.shape[0])
            for away in range(score_matrix.shape[1])
        ]
    )
    .nlargest(10, 'probability')
    .reset_index(drop=True)
)
top_scorelines['probability'] = top_scorelines['probability'].map('{:.2%}'.format)
display(top_scorelines)

## 4. Simulate the rest of the season

The model trains on the complete historical window, but league points and positions use only the current season. The remaining fixtures are sampled 2,000 times.

In [ ]:
latest_season = completed['season'].iloc[-1]
current_results = completed.loc[completed['season'].eq(latest_season)]
display(current_table(current_results))

simulation_summary, position_probabilities = simulate_league(
    model,
    completed,
    remaining_fixtures,
    n_simulations=2_000,
    seed=42,
)
formatted_simulation = simulation_summary.copy()
formatted_simulation['expected_points'] = formatted_simulation['expected_points'].round(1)
formatted_simulation['expected_position'] = formatted_simulation['expected_position'].round(2)
for column in ['title_probability', 'top_four_probability', 'relegation_probability']:
    formatted_simulation[column] = formatted_simulation[column].map('{:.1%}'.format)
display(formatted_simulation)
simulation_summary.to_csv(REPORT_DIR / 'synthetic_simulation_summary.csv', index=False)

In [ ]:
selected_team = simulation_summary.iloc[0]['team']
distribution = position_probabilities.loc[
    position_probabilities['team'].eq(selected_team)
]
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(distribution['position'], distribution['probability'])
ax.set(
    xlabel='Final position',
    ylabel='Probability',
    title=f'Finish distribution — {selected_team}',
)
ax.set_xticks(distribution['position'])
ax.grid(axis='y', alpha=0.25)
plt.show()

## 5. Optional: real Premier League and La Liga data

Change `RUN_REAL_DATA` to `True` to download five seasons from Football-Data.co.uk. No API key is required. This may take several minutes because the models are repeatedly fitted through time. Provider files remain local and are excluded from Git.

In [ ]:
from football_lab.data.football_data_uk import fetch_many
from football_lab.pipeline import run_real_backtest

RUN_REAL_DATA = False

if RUN_REAL_DATA:
    real_matches = fetch_many(
        leagues=['premier_league', 'la_liga'],
        seasons=['2021-22', '2022-23', '2023-24', '2024-25', '2025-26'],
        output_dir='data/raw/football_data_uk',
    )
    real_matches.to_csv('data/matches.csv', index=False)
    display(real_matches.groupby(['league', 'season']).size().rename('matches').to_frame())

    real_metrics = run_real_backtest(
        real_matches,
        output_dir='reports/real_data',
        initial_train_matches=760,
        refit_every=100,
    )
    display(real_metrics.round(4))
else:
    print('Real-data download is disabled. Set RUN_REAL_DATA = True when ready.')

## Interpretation checklist

Before publishing real results, verify that: (1) every prediction predates kickoff, (2) EPL and La Liga are evaluated separately, (3) model log loss is compared with normalized market probabilities, (4) uncertainty intervals are added for model differences, and (5) synthetic results are never presented as real performance.